# Mastering LLM Deployment
## Lab 4 - Model Pruning

**Duration:** ~90 minutes  ·  **Runtime:** T4 GPU  ·  **Prerequisite:** Lab 1 (`models/teacher-bert-sst2/`)

---

### The lever with the biggest gap between theory and practice

Pruning removes weights outright. The literature routinely reports 80–90% of a transformer's weights removed with modest quality loss, which sounds like the best of the three levers by a wide margin.

It usually is not, and the reason is the single most important thing in this lab:

> **Setting a weight to zero does not make the multiplication disappear.** A dense GEMM kernel on a GPU or CPU multiplies by zero at exactly the same speed it multiplies by anything else. Unstructured sparsity gives you a *compressible* model, not a *faster* one - unless your serving stack has a sparse kernel that exploits the specific sparsity pattern you produced.

That is why the Lab 1 case study's playbook reached for distillation and quantization, both of which change what the hardware actually executes. Pruning still earns its place - for storage, for transfer, and above all in its **structured** form - but only if you know which kind you are doing and why.

### Learning outcomes

- Distinguish unstructured from structured pruning and predict which yields real latency gains.
- Apply magnitude pruning with `tensorflow_model_optimization`: sparsity schedules, the pruning step callback, and stripping the wrappers for export.
- Explain why the TFMOT wrapper cannot be applied to a Hugging Face transformer, and implement mask-based pruning yourself instead.
- Produce a sparsity-versus-accuracy curve, with and without fine-tuning, and identify the knee.
- Perform **structured** feed-forward pruning that physically shrinks the model and measure the resulting speedup.
- Add pruning rows to the ledger.

**Expected GPU time: 10–14 minutes.**

---
## 0. Environment setup

In [ ]:
%pip install -q "transformers>=4.40,<5" "datasets>=2.19,<4" "tf-keras>=2.16" "tensorflow-model-optimization>=0.8.0" "scikit-learn" "pandas"
print('dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
dependencies installed


In [ ]:
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
if "tensorflow" in sys.modules:
    print("TensorFlow already imported -> Runtime > Restart session and re-run from the top.")

import tensorflow as tf, numpy as np, pandas as pd, time, gzip, shutil
# tf.keras is a lazy loader and does not re-export __version__.
try:
    import tf_keras as _keras_pkg
except ImportError:
    import keras as _keras_pkg
_keras_impl = tf.keras.Model.__module__
print("TF", tf.__version__, "| Keras", _keras_pkg.__version__, "|", _keras_impl)
assert _keras_pkg.__version__.startswith("2.") and "tf_keras" in _keras_impl, (
    "Keras 2 is not active. Install tf-keras, set TF_USE_LEGACY_KERAS=1 before "
    "importing tensorflow, then Runtime > Restart session.")
tf.keras.utils.set_random_seed(42)

TF 2.20.0 | Keras 2.20.0 | tf_keras.src.engine.training


In [ ]:
USE_DRIVE = True
ROOT = "/content/llm-deploy-labs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/llm-deploy-labs"
    except Exception as e:
        print("Drive unavailable:", e)
os.environ["LLMDEPLOY_ROOT"] = ROOT
for sub in ("models", "reports", "data"):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print("Artifact root:", ROOT)

Mounted at /content/drive
Artifact root: /content/drive/MyDrive/llm-deploy-labs


In [ ]:
LABKIT_SRC = r'''
"""
labkit.py - shared utilities for the "Mastering LLM Deployment" hands-on labs.

Everything the labs need in common lives here so that each notebook measures
the same things in the same way:

  * artifact + ledger management (results survive across notebooks via Drive)
  * a model "size on disk" and parameter/sparsity accounting
  * a latency/throughput benchmark harness with warm-up and percentiles
  * a minimal, explicit GradientTape training loop (works for HF TF models,
    plain Keras models, distillation losses and masked/pruned training alike)
"""

import os

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# --------------------------------------------------------------------------
# 1. Artifact root
# --------------------------------------------------------------------------

_ROOT = Path(os.environ.get("LLMDEPLOY_ROOT", "/content/llm-deploy-labs"))


def set_root(path):
    """Point the lab kit at a persistent directory (ideally on Google Drive)."""
    global _ROOT
    _ROOT = Path(path)
    for sub in ("models", "reports", "data"):
        (_ROOT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["LLMDEPLOY_ROOT"] = str(_ROOT)
    return _ROOT


def root():
    return _ROOT


def model_dir(name, clean=False):
    """Return (and create) a directory under <root>/models/<name>."""
    d = _ROOT / "models" / name
    if clean and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# 2. Size and parameter accounting
# --------------------------------------------------------------------------


def size_mb(path):
    """Size of a file or, recursively, of a directory - in MB."""
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6


def count_params(model):
    """Total trainable parameter count."""
    return int(sum(int(np.prod(v.shape)) for v in model.trainable_variables))


def weight_sparsity(model, kinds=("kernel", "weight", "embeddings")):
    """Fraction of zeros across the "real" weight matrices (ignores biases /
    LayerNorm, which are never pruned in practice)."""
    zeros, total = 0, 0
    for v in model.trainable_variables:
        if not any(k in v.name for k in kinds):
            continue
        arr = v.numpy()
        zeros += int((arr == 0).sum())
        total += int(arr.size)
    return zeros / max(total, 1)


# --------------------------------------------------------------------------
# 3. Latency / throughput benchmarking
# --------------------------------------------------------------------------


def measure_latency(predict_fn, inputs, warmup=5, runs=30, batch_size=1):
    """Run predict_fn(inputs) repeatedly and report wall-clock percentiles.

    Warm-up matters: the first calls pay for graph tracing, kernel autotuning
    and (on GPU) cuDNN algorithm selection. Reporting those numbers is the
    single most common benchmarking mistake in deployment work.
    """
    for _ in range(warmup):
        predict_fn(inputs)

    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        predict_fn(inputs)
        samples.append((time.perf_counter() - t0) * 1000.0)

    samples = np.array(sorted(samples))
    p50 = float(np.percentile(samples, 50))
    return {
        "mean_ms": round(float(samples.mean()), 2),
        "p50_ms": round(p50, 2),
        "p90_ms": round(float(np.percentile(samples, 90)), 2),
        "p95_ms": round(float(np.percentile(samples, 95)), 2),
        "throughput_rps": round(batch_size / (p50 / 1000.0), 1),
    }


def device_label():
    return "GPU" if tf.config.list_physical_devices("GPU") else "CPU"


# --------------------------------------------------------------------------
# 4. The optimization ledger
# --------------------------------------------------------------------------


def _ledger_file():
    (_ROOT / "reports").mkdir(parents=True, exist_ok=True)
    return _ROOT / "reports" / "ledger.json"


def load_ledger():
    f = _ledger_file()
    if not f.exists():
        return []
    return json.loads(f.read_text())


def record(stage, **fields):
    """Insert or replace a ledger row. Stage names are unique keys, so
    re-running a cell updates the row instead of duplicating it."""
    ledger = [e for e in load_ledger() if e.get("stage") != stage]
    entry = {"stage": stage, "recorded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(fields)
    ledger.append(entry)
    _ledger_file().write_text(json.dumps(ledger, indent=2))
    return entry


def ledger_df(columns=None):
    import pandas as pd

    df = pd.DataFrame(load_ledger())
    if df.empty:
        return df
    preferred = [
        "stage",
        "model",
        "task",
        "dataset",
        "params_m",
        "size_mb",
        "quality",
        "quality_metric",
        "p50_ms",
        "p95_ms",
        "throughput_rps",
        "device",
        "notes",
    ]
    cols = columns or [c for c in preferred if c in df.columns]
    extra = [c for c in df.columns if c not in cols and c != "recorded_at"]
    return df[cols + extra]


# --------------------------------------------------------------------------
# 5. A small, explicit training loop
# --------------------------------------------------------------------------


def train(
    model,
    dataset,
    loss_fn,
    optimizer,
    epochs=1,
    steps_per_epoch=None,
    log_every=50,
    on_step_end=None,
    clip_norm=1.0,
):
    """Generic GradientTape loop.

    loss_fn(model, batch, training) -> scalar loss tensor.
    on_step_end(global_step) -> optional Python callback, used by the pruning
    lab to update sparsity masks between steps.
    """

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, batch, True)
        grads = tape.gradient(loss, model.trainable_variables)
        pairs = [
            (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
        ]
        if clip_norm:
            gs, _ = tf.clip_by_global_norm([g for g, _ in pairs], clip_norm)
            pairs = list(zip(gs, [v for _, v in pairs]))
        optimizer.apply_gradients(pairs)
        return loss

    global_step = 0
    history = []
    for epoch in range(epochs):
        running, seen = 0.0, 0
        t0 = time.time()
        for step, batch in enumerate(dataset):
            loss = float(train_step(batch))
            running += loss
            seen += 1
            global_step += 1
            if on_step_end is not None:
                on_step_end(global_step)
            if log_every and global_step % log_every == 0:
                print(
                    f"  epoch {epoch + 1} | step {global_step:>5} | "
                    f"loss {running / seen:.4f}"
                )
                running, seen = 0.0, 0
            if steps_per_epoch and step + 1 >= steps_per_epoch:
                break
        history.append({"epoch": epoch + 1, "seconds": round(time.time() - t0, 1)})
        print(f"  epoch {epoch + 1} finished in {history[-1]['seconds']}s")
    return history


# --------------------------------------------------------------------------
# 6. Evaluation helpers
# --------------------------------------------------------------------------


def evaluate_accuracy(logits_fn, dataset):
    """logits_fn(features) -> array of shape [batch, num_classes]."""
    correct, total = 0, 0
    for features, labels in dataset:
        logits = np.asarray(logits_fn(features))
        preds = logits.argmax(axis=-1)
        labels = np.asarray(labels)
        correct += int((preds == labels).sum())
        total += int(labels.shape[0])
    return correct / max(total, 1)


def hf_logits_fn(model):
    """Wrap a Hugging Face TF model so it returns a plain logits tensor and is
    compiled once into a graph (fair, low-overhead benchmarking)."""

    @tf.function(reduce_retracing=True)
    def fn(features):
        return model(features, training=False).logits

    return fn


def banner(title):
    line = "=" * max(60, len(title) + 4)
    print(f"\n{line}\n  {title}\n{line}")
'''

from pathlib import Path
Path(ROOT, 'labkit.py').write_text(LABKIT_SRC)

import sys, importlib
sys.path.insert(0, ROOT)
import labkit as lk
importlib.reload(lk)
lk.set_root(ROOT)
lk.banner('lab kit ready')
print('ledger rows:', [r['stage'] for r in lk.load_ledger()])


  lab kit ready
ledger rows: ['baseline', 'squad-teacher', 'distilled-4L', 'quantized-int8', 'quantized-fp16']


---
## 1. How pruning works

### 1.1 Two families

| | **Unstructured** | **Structured** |
|---|---|---|
| Removes | individual weights | whole rows, columns, neurons, attention heads, layers |
| Resulting tensor | same shape, mostly zeros | genuinely smaller |
| Achievable sparsity | very high (80–95%) | modest (20–50%) |
| Disk size | shrinks a lot **after compression** | shrinks directly |
| Memory at run time | unchanged (dense storage) | shrinks |
| **Latency on standard hardware** | **unchanged** | **improves, roughly proportionally** |
| Needs special kernels | yes, to see any speed benefit | no |

The middle ground is **semi-structured sparsity**, most commonly the 2:4 pattern - exactly two zeros in every group of four contiguous weights. NVIDIA Ampere and later have sparse tensor cores that execute this pattern at up to twice the dense rate. It is the only unstructured-looking sparsity that reliably speeds anything up, and it requires both hardware support and a runtime that emits the sparse kernel.

### 1.2 Magnitude pruning

The workhorse criterion: rank weights by $|w|$ and zero the smallest fraction. It is a first-order approximation of "which weights contribute least to the output", and despite far more sophisticated alternatives (Optimal Brain Surgeon, movement pruning, Fisher-information scoring) it remains competitive and is trivial to implement.

One decision matters more than the criterion: **global versus per-layer thresholds.** A global threshold ranks every weight in the model together, which lets naturally low-magnitude layers be pruned harder - often good, but it can empty a layer entirely and break the network. A per-layer threshold applies the same sparsity everywhere, which is safer and more predictable. We use per-layer here and let you compare in the exercises.

### 1.3 One-shot vs gradual, and why fine-tuning is mandatory

Pruning to a high sparsity in one step and stopping is the worst way to do it. The standard recipe is **gradual magnitude pruning**: raise sparsity from 0 to the target over many steps while training continues, so the surviving weights can compensate for the removed ones. A common schedule is cubic:

$$s_t = s_f\left(1 - \left(1 - \frac{t}{T}\right)^3\right)$$

which prunes aggressively early - when the model has plenty of redundancy - and slowly near the target, when each removal costs more.

### 1.4 What actually gets pruned in a transformer

- **Feed-forward layers** are the best target: roughly two-thirds of the non-embedding parameters, and highly redundant.
- **Attention projections** are prunable, but the head structure is a more effective unit - many heads are measurably redundant, and removing a whole head is *structured* pruning with real speedup.
- **Embeddings should generally not be pruned.** Zeroing entries in a lookup table degrades rare tokens specifically, which is invisible in aggregate metrics and painful in production. Quantize embeddings instead.
- **LayerNorm and biases** are far too small to be worth touching and are numerically sensitive.

---
## 2. The standard API, on a model it supports

`tensorflow_model_optimization` wraps each prunable layer with a mask, a schedule, and a step counter. The API is short, and you should know it because it is what production TensorFlow pipelines use.

It also has a hard requirement: `prune_low_magnitude` **clones the model graph**, so it needs a Sequential or Functional model. A Hugging Face `TFBertForSequenceClassification` is a *subclassed* model - as we saw in Lab 1 1.3, there is no static graph to clone, and the call fails. Section 3 shows what to do instead.

We build the supported case first: a Functional classifier over SST-2, per the syllabus.

In [ ]:
from datasets import load_dataset

sst2 = load_dataset("nyu-mll/glue", "sst2")
VOCAB, N_BOW = 3_000, 10_000

train_txt = sst2["train"].shuffle(seed=42).select(range(N_BOW))
val_txt   = sst2["validation"]

vectorizer = tf.keras.layers.TextVectorization(max_tokens=VOCAB, output_mode="multi_hot")
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(train_txt["sentence"]).batch(256))

Xtr = vectorizer(tf.constant(train_txt["sentence"])).numpy().astype(np.float32)
Ytr = np.asarray(train_txt["label"], np.int32)
Xva = vectorizer(tf.constant(val_txt["sentence"])).numpy().astype(np.float32)
Yva = np.asarray(val_txt["label"], np.int32)
print("train", Xtr.shape, "| val", Xva.shape)

inp = tf.keras.Input(shape=(VOCAB,), name="tokens")
h = tf.keras.layers.Dense(256, activation="relu")(inp)
h = tf.keras.layers.Dense(128, activation="relu")(h)
out = tf.keras.layers.Dense(2, name="logits")(h)
dense_model = tf.keras.Model(inp, out, name="sst2_bow")
dense_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=["accuracy"])
dense_model.fit(Xtr, Ytr, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
dense_acc = dense_model.evaluate(Xva, Yva, verbose=0)[1]
print(f"\ndense baseline accuracy: {dense_acc:.4f} | params {dense_model.count_params():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

train (10000, 3000) | val (872, 3000)
Epoch 1/5
71/71 - 2s - loss: 0.5791 - accuracy: 0.6754 - val_loss: 0.4343 - val_accuracy: 0.7820 - 2s/epoch - 28ms/step
Epoch 2/5
71/71 - 0s - loss: 0.3091 - accuracy: 0.8583 - val_loss: 0.3897 - val_accuracy: 0.8250 - 316ms/epoch - 4ms/step
Epoch 3/5
71/71 - 0s - loss: 0.1783 - accuracy: 0.9234 - val_loss: 0.4568 - val_accuracy: 0.8200 - 298ms/epoch - 4ms/step
Epoch 4/5
71/71 - 0s - loss: 0.1183 - accuracy: 0.9434 - val_loss: 0.5237 - val_accuracy: 0.8170 - 280ms/epoch - 4ms/step
Epoch 5/5
71/71 - 0s - loss: 0.0887 - accuracy: 0.9537 - val_loss: 0.6301 - val_accuracy: 0.8120 - 287ms/epoch - 4ms/step

dense baseline accuracy: 0.7592 | params 801,410


In [ ]:
import tensorflow_model_optimization as tfmot
prune = tfmot.sparsity.keras

EPOCHS_PRUNE, BATCH = 5, 128
steps = int(np.ceil(len(Xtr) * 0.9 / BATCH)) * EPOCHS_PRUNE

pruning_params = {
    "pruning_schedule": prune.PolynomialDecay(
        initial_sparsity=0.20,
        final_sparsity=0.85,
        begin_step=0,
        end_step=steps,
        power=3,            # cubic: prune fast early, slowly near the target
    )
}
pruned = prune.prune_low_magnitude(dense_model, **pruning_params)
pruned.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
               metrics=["accuracy"])

# UpdatePruningStep is not optional: it advances the schedule's step counter.
# Without it the masks never change and sparsity stays at initial_sparsity.
pruned.fit(Xtr, Ytr, epochs=EPOCHS_PRUNE, batch_size=BATCH, validation_split=0.1,
           verbose=2, callbacks=[prune.UpdatePruningStep()])

stripped = prune.strip_pruning(pruned)     # remove masks/counters for export
stripped.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=["accuracy"])
pruned_acc = stripped.evaluate(Xva, Yva, verbose=0)[1]

zeros = sum(int((w == 0).sum()) for w in stripped.get_weights() if w.ndim == 2)
total = sum(int(w.size) for w in stripped.get_weights() if w.ndim == 2)
print(f"\ndense  {dense_acc:.4f} | pruned {pruned_acc:.4f} "
      f"({pruned_acc - dense_acc:+.4f}) | sparsity {zeros/total:.1%}")

### 2.1 Where the size win actually shows up

Save both models and compare - first raw, then gzipped. This is the cell that makes the central point of the lab concrete.

In [ ]:
prune_dir = os.path.join(ROOT, "models", "pruning")
os.makedirs(prune_dir, exist_ok=True)

def save_and_gzip(model, name):
    h5 = os.path.join(prune_dir, f"{name}.h5")
    model.save(h5, include_optimizer=False)
    gz = h5 + ".gz"
    with open(h5, "rb") as src, gzip.open(gz, "wb", compresslevel=9) as dst:
        shutil.copyfileobj(src, dst)
    return lk.size_mb(h5), lk.size_mb(gz)

d_raw, d_gz = save_and_gzip(dense_model, "dense")
p_raw, p_gz = save_and_gzip(stripped, "pruned85")

print(f"{'':<10}{'raw MB':>10}{'gzipped MB':>14}")
print(f"{'dense':<10}{d_raw:>10.3f}{d_gz:>14.3f}")
print(f"{'pruned':<10}{p_raw:>10.3f}{p_gz:>14.3f}")
print(f"\nraw size ratio     : {p_raw/d_raw:.3f}  <- essentially unchanged")
print(f"gzipped size ratio : {p_gz/d_gz:.3f}  <- this is the real pruning win")

In [ ]:
# And the part people expect but do not get: latency.
sample = tf.constant(Xva[:1])
dense_fn = tf.function(lambda x: dense_model(x, training=False))
sparse_fn = tf.function(lambda x: stripped(x, training=False))

d_lat = lk.measure_latency(dense_fn, sample, warmup=20, runs=200, batch_size=1)
s_lat = lk.measure_latency(sparse_fn, sample, warmup=20, runs=200, batch_size=1)
print("dense :", d_lat)
print("sparse:", s_lat)
print(f"\nspeedup from 85% sparsity: {d_lat['p50_ms']/s_lat['p50_ms']:.2f}x")
print("Expect roughly 1.0x. The zeros are still stored and still multiplied.")

dense : {'mean_ms': 0.62, 'p50_ms': 0.62, 'p90_ms': 0.67, 'p95_ms': 0.7, 'throughput_rps': 1615.9}
sparse: {'mean_ms': 0.66, 'p50_ms': 0.64, 'p90_ms': 0.75, 'p95_ms': 0.82, 'throughput_rps': 1566.5}

speedup from 85% sparsity: 0.97x
Expect roughly 1.0x. The zeros are still stored and still multiplied.


Three conclusions to write down:

1. **Raw file size barely moves.** The zeros occupy the same 4 bytes each.
2. **Compressed size drops substantially** - long runs of zeros are exactly what a general-purpose compressor exploits. This is a genuine and useful benefit: smaller container images, faster model downloads, cheaper artifact storage, faster cold starts on ECS. Day 2 will care about this.
3. **Latency is unchanged.** If you want pruning to make inference faster, you need structured pruning (Section 4), a 2:4 pattern on supporting hardware, or a sparse runtime.

---
## 3. Pruning the real transformer

### 3.1 Why we implement it ourselves

Run the next cell to see the failure first-hand. `prune_low_magnitude` needs to walk and rebuild a layer graph; a subclassed model does not expose one.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

TEACHER_DIR = os.path.join(ROOT, "models", "teacher-bert-sst2")
if not os.path.isdir(TEACHER_DIR):
    raise FileNotFoundError(f"{TEACHER_DIR} not found - run Lab 1 Section 4 first.")

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)
bert = TFAutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
_ = bert(bert.dummy_inputs, training=False)

try:
    prune.prune_low_magnitude(bert)
    print("Unexpected: the wrapper accepted a subclassed model.")
except Exception as e:
    print(f"prune_low_magnitude failed as expected:\n  {type(e).__name__}: "
          f"{str(e)[:220]}")
    print("\n-> We implement masking directly on the variables instead.")

In [ ]:
MAX_LEN = 128

def encode(texts):
    enc = tokenizer(list(texts), max_length=MAX_LEN, truncation=True,
                    padding="max_length", return_tensors="np")
    return {k: np.asarray(v, np.int32) for k, v in enc.items()}

def make_ds(texts, labels, bs=64, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((encode(texts), np.asarray(labels, np.int32)))
    if shuffle:
        ds = ds.shuffle(4096, seed=42)
    return ds.batch(bs).prefetch(tf.data.AUTOTUNE)

val_ds = make_ds(val_txt["sentence"], val_txt["label"])
ft_ds  = make_ds(train_txt["sentence"], train_txt["label"], bs=32, shuffle=True)

bert_fn = lk.hf_logits_fn(bert)
bert_acc = lk.evaluate_accuracy(bert_fn, val_ds)
print(f"BERT baseline accuracy: {bert_acc:.4f}")

BERT baseline accuracy: 0.9140


### 3.2 Mask-based magnitude pruning

The implementation is 20 lines. We target only 2-D kernels inside the encoder - feed-forward and attention projections - and deliberately exclude the embedding table, the pooler and the classifier head.

In [ ]:
ORIGINALS = {v.name: v.numpy().copy() for v in bert.weights}

def prunable_variables(model):
    out = []
    for v in model.trainable_variables:
        name = v.name.lower()
        if len(v.shape) != 2 or "kernel" not in name:
            continue
        if "embeddings" in name or "pooler" in name or "classifier" in name:
            continue
        out.append(v)
    return out

TARGETS = prunable_variables(bert)
print(f"{len(TARGETS)} prunable matrices, "
      f"{sum(int(np.prod(v.shape)) for v in TARGETS)/1e6:.1f}M parameters "
      f"({sum(int(np.prod(v.shape)) for v in TARGETS)/bert.num_parameters():.0%} of the model)")

def restore():
    for v in bert.weights:
        v.assign(ORIGINALS[v.name])

def compute_masks(sparsity):
    # Per-layer magnitude threshold: keep the largest (1 - sparsity) fraction.
    masks = {}
    for v in TARGETS:
        w = np.abs(ORIGINALS[v.name])
        k = int(round(sparsity * w.size))
        if k <= 0:
            masks[v.name] = np.ones_like(w, np.float32)
            continue
        thresh = np.partition(w.ravel(), k - 1)[k - 1]
        masks[v.name] = (w > thresh).astype(np.float32)
    return masks

def apply_masks(masks):
    for v in TARGETS:
        v.assign(v * masks[v.name])

In [ ]:
lk.banner("one-shot pruning: no fine-tuning")
one_shot = []
for s in (0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    restore()
    if s > 0:
        apply_masks(compute_masks(s))
    acc = lk.evaluate_accuracy(lk.hf_logits_fn(bert), val_ds)
    one_shot.append({"sparsity": s, "accuracy": round(acc, 4),
                     "delta": round(acc - bert_acc, 4)})
    print(f"  sparsity {s:>4.0%}  accuracy {acc:.4f}  ({acc - bert_acc:+.4f})")
restore()
pd.DataFrame(one_shot)


  one-shot pruning: no fine-tuning
  sparsity   0%  accuracy 0.9140  (+0.0000)
  sparsity  20%  accuracy 0.9071  (-0.0069)
  sparsity  40%  accuracy 0.8922  (-0.0218)
  sparsity  50%  accuracy 0.8567  (-0.0573)
  sparsity  60%  accuracy 0.7523  (-0.1617)
  sparsity  70%  accuracy 0.5103  (-0.4037)
  sparsity  80%  accuracy 0.5103  (-0.4037)
  sparsity  90%  accuracy 0.5034  (-0.4106)


,sparsity,accuracy,delta
0,0.0,0.9140,0.0000
1,0.2,0.9071,-0.0069
2,0.4,0.8922,-0.0218
3,0.5,0.8567,-0.0573
4,0.6,0.7523,-0.1617
5,0.7,0.5103,-0.4037
6,0.8,0.5103,-0.4037
7,0.9,0.5034,-0.4106


The curve is flat, then it falls off a cliff. That shape is the signature of magnitude pruning on an over-parameterised network: a large fraction of weights genuinely contributes nothing, and past that point you start removing weights the model needs.

Where your knee sits is the number that matters. Everything below it is free capacity you were paying to store and ship.

### 3.3 Gradual pruning with fine-tuning

Now the proper recipe. We raise sparsity along a cubic schedule while training continues, re-applying masks after every optimizer step so pruned weights cannot drift back from zero.

The `on_step_end` hook in `lk.train` exists for exactly this.

**Expected time: 5–10 minutes.**

In [13]:
TARGET_SPARSITY = 0.70
FT_STEPS        = 400
RECOMPUTE_EVERY = 25

restore()
state = {"masks": compute_masks(0.0), "sparsity": 0.0}

def cubic_sparsity(step, total, final):
    t = min(step / max(total, 1), 1.0)
    return final * (1.0 - (1.0 - t) ** 3)

def pruning_callback(step):
    if step % RECOMPUTE_EVERY == 0 or step == 1:
        s = cubic_sparsity(step, FT_STEPS, TARGET_SPARSITY)
        # Rank by the *current* weights, not the originals, so the schedule
        # reflects what fine-tuning has done so far.
        masks = {}
        for v in TARGETS:
            w = np.abs(v.numpy())
            k = int(round(s * w.size))
            masks[v.name] = (np.ones_like(w, np.float32) if k <= 0
                             else (w > np.partition(w.ravel(), k - 1)[k - 1]).astype(np.float32))
        state["masks"], state["sparsity"] = masks, s
    apply_masks(state["masks"])

scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def clf_loss(model, batch, training):
    features, labels = batch
    return scce(labels, model(features, training=training).logits)

sched = tf.keras.optimizers.schedules.PolynomialDecay(2e-5, FT_STEPS, end_learning_rate=0.0)
opt = tf.keras.optimizers.Adam(learning_rate=sched)

lk.banner(f"gradual pruning to {TARGET_SPARSITY:.0%} over {FT_STEPS} steps")
lk.train(bert, ft_ds.repeat(), clf_loss, opt, epochs=1,
         steps_per_epoch=FT_STEPS, log_every=100, on_step_end=pruning_callback)

apply_masks(state["masks"])
gradual_acc = lk.evaluate_accuracy(lk.hf_logits_fn(bert), val_ds)
achieved = lk.weight_sparsity(bert, kinds=("kernel",))

one_shot_at_target = next(r for r in one_shot if abs(r["sparsity"] - TARGET_SPARSITY) < 1e-6)
print(f"\nbaseline dense              : {bert_acc:.4f}")
print(f"one-shot at {TARGET_SPARSITY:.0%}            : {one_shot_at_target['accuracy']:.4f}")
print(f"gradual + fine-tune at {TARGET_SPARSITY:.0%} : {gradual_acc:.4f}")
print(f"recovered by fine-tuning    : "
      f"{gradual_acc - one_shot_at_target['accuracy']:+.4f}")
print(f"measured kernel sparsity    : {achieved:.1%}")


  gradual pruning to 70% over 400 steps
  epoch 1 | step   100 | loss 0.1561
  epoch 1 | step   200 | loss 0.1868
  epoch 1 | step   300 | loss 0.3515
  epoch 1 | step   400 | loss 0.3354
  epoch 1 finished in 448.2s

baseline dense              : 0.9140
one-shot at 70%            : 0.5103
gradual + fine-tune at 70% : 0.8119
recovered by fine-tuning    : +0.3016
measured kernel sparsity    : 69.5%


In [14]:
sparse_dir = lk.model_dir("bert-sst2-pruned70", clean=True)
bert.save_pretrained(sparse_dir)
tokenizer.save_pretrained(sparse_dir)

# Compressed size is where unstructured sparsity pays.
h5 = os.path.join(sparse_dir, "tf_model.h5")
gz = h5 + ".gz"
if os.path.exists(h5):
    with open(h5, "rb") as src, gzip.open(gz, "wb", compresslevel=6) as dst:
        shutil.copyfileobj(src, dst)
    print(f"sparse model raw      : {lk.size_mb(h5):.1f} MB")
    print(f"sparse model gzipped  : {lk.size_mb(gz):.1f} MB "
          f"({lk.size_mb(gz)/lk.size_mb(h5):.2f}x)")

sp_lat = lk.measure_latency(lk.hf_logits_fn(bert),
                            {k: tf.constant(v[:1]) for k, v in encode(val_txt["sentence"]).items()},
                            warmup=10, runs=50, batch_size=1)

lk.record("pruned-unstructured-70", model="BERT-base, 70% sparse kernels",
          task="binary sentiment", dataset="SST-2",
          params_m=round(bert.num_parameters()/1e6, 1),
          size_mb=round(lk.size_mb(gz) if os.path.exists(gz) else lk.size_mb(sparse_dir), 1),
          quality=round(gradual_acc, 4), quality_metric="accuracy",
          p50_ms=sp_lat["p50_ms"], p95_ms=sp_lat["p95_ms"],
          throughput_rps=sp_lat["throughput_rps"], device=lk.device_label(),
          notes="gzipped size; latency unchanged (dense kernels)")
lk.ledger_df()

sparse model raw      : 438.2 MB
sparse model gzipped  : 221.6 MB (0.51x)


,stage,model,task,dataset,params_m,size_mb,quality,quality_metric,p50_ms,p95_ms,throughput_rps,device,notes,p50_ms_batch32,imdb_accuracy
0,baseline,bert-base-uncased (fine-tuned),binary sentiment,SST-2,109.5,439.2,0.9140,accuracy,26.96,31.34,37.1,GPU,teacher; fixed 128-token padding,291.02,NaN
1,squad-teacher,bert-base-uncased (SQuAD v1.1),extractive QA,SQuAD v1.1,108.9,435.8,89.8700,F1,30.03,31.52,33.3,GPU,12 layers; distillation teacher,NaN,NaN
2,distilled-4L,BERT 4-layer student,extractive QA,SQuAD v1.1,52.2,209.8,54.8200,F1,10.17,11.68,98.3,GPU,"T=3.0, alpha=0.7, layers copied [2, 5, 8, 11]",NaN,NaN
3,quantized-int8,"BERT-base, int8 per-channel kernels",binary sentiment,SST-2 / IMDB,109.5,181.7,0.9106,accuracy,27.82,40.08,35.9,GPU,size projected from format; accuracy measured;...,NaN,0.8265
4,quantized-fp16,"BERT-base, fp16 weights",binary sentiment,SST-2 / IMDB,109.5,219.0,0.9140,accuracy,27.82,40.08,35.9,GPU,fp16 is typically lossless for BERT-scale infe...,NaN,NaN
5,pruned-unstructured-70,"BERT-base, 70% sparse kernels",binary sentiment,SST-2,109.5,221.6,0.8119,accuracy,11.16,15.15,89.6,GPU,gzipped size; latency unchanged (dense kernels),NaN,NaN


---
## 4. Structured pruning: the version that actually gets faster

Everything so far produced zeros inside tensors of unchanged shape. Structured pruning removes the shape itself.

The feed-forward block is the natural target. Each encoder layer expands 768 → 3072, applies GELU, and projects back 3072 → 768. Intermediate neuron $j$ is used by exactly one column of the up-projection and one row of the down-projection, so removing it means deleting that column and that row - leaving a smaller, fully dense, ordinary model that every runtime executes faster with no special support.

**Importance score.** We use the product of the two norms attached to each neuron:

$$\mathrm{importance}(j) = \lVert W^{\text{in}}_{:,j} \rVert_2 \cdot \lVert W^{\text{out}}_{j,:} \rVert_2$$

A neuron matters only if it both receives and transmits signal; a large incoming weight feeding a near-zero outgoing weight contributes nothing.

In [15]:
from transformers import BertConfig, TFBertForSequenceClassification

KEEP_RATIO = 0.5            # 3072 -> 1536 intermediate units per layer

restore()                   # start structured pruning from the dense baseline
src_cfg = bert.config
new_cfg = BertConfig.from_dict(src_cfg.to_dict())
new_cfg.intermediate_size = int(src_cfg.intermediate_size * KEEP_RATIO)

slim = TFBertForSequenceClassification(new_cfg)
_ = slim(bert.dummy_inputs, training=False)

def ffn_importance(layer):
    w_in, _ = layer.intermediate.dense.get_weights()      # [768, 3072]
    w_out, _ = layer.bert_output.dense.get_weights()      # [3072, 768]
    return np.linalg.norm(w_in, axis=0) * np.linalg.norm(w_out, axis=1)

try:
    slim.bert.embeddings.set_weights(bert.bert.embeddings.get_weights())
    kept_per_layer = []
    for i, (src, dst) in enumerate(zip(bert.bert.encoder.layer, slim.bert.encoder.layer)):
        dst.attention.set_weights(src.attention.get_weights())

        imp = ffn_importance(src)
        keep = np.sort(np.argsort(imp)[-new_cfg.intermediate_size:])
        kept_per_layer.append(keep)

        w_in, b_in = src.intermediate.dense.get_weights()
        dst.intermediate.dense.set_weights([w_in[:, keep], b_in[keep]])

        w_out, b_out = src.bert_output.dense.get_weights()
        dst.bert_output.dense.set_weights([w_out[keep, :], b_out])
        dst.bert_output.LayerNorm.set_weights(src.bert_output.LayerNorm.get_weights())

    slim.bert.pooler.set_weights(bert.bert.pooler.get_weights())
    slim.classifier.set_weights(bert.classifier.get_weights())
    print("structured student assembled from the dense teacher")
except Exception as e:
    print("Structured surgery failed:", type(e).__name__, e)
    raise

print(f"dense      {bert.num_parameters()/1e6:6.1f}M parameters")
print(f"structured {slim.num_parameters()/1e6:6.1f}M parameters "
      f"({slim.num_parameters()/bert.num_parameters():.0%})")

structured student assembled from the dense teacher
dense       109.5M parameters
structured   81.2M parameters (74%)


In [16]:
slim_acc_before = lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds)
print(f"accuracy immediately after surgery: {slim_acc_before:.4f} "
      f"(baseline {bert_acc:.4f})")
print("A drop here is expected - the remaining neurons have not yet adjusted.")

accuracy immediately after surgery: 0.5092 (baseline 0.9140)
A drop here is expected - the remaining neurons have not yet adjusted.


### 4.1 Recover with fine-tuning

Structured pruning removes coordinated groups of weights, so the damage is larger than unstructured pruning at comparable parameter counts - and correspondingly more recoverable, because the surviving network is a normal dense model that trains normally.

**Expected time: 2–3 minutes.**

In [17]:
RECOVER_STEPS = 400
sched = tf.keras.optimizers.schedules.PolynomialDecay(3e-5, RECOVER_STEPS, end_learning_rate=0.0)
opt = tf.keras.optimizers.Adam(learning_rate=sched)

lk.banner("fine-tuning the structurally pruned model")
lk.train(slim, ft_ds.repeat(), clf_loss, opt, epochs=1,
         steps_per_epoch=RECOVER_STEPS, log_every=100)

slim_acc = lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds)
print(f"\nafter surgery      {slim_acc_before:.4f}")
print(f"after fine-tuning  {slim_acc:.4f}  (baseline {bert_acc:.4f}, "
      f"delta {slim_acc - bert_acc:+.4f})")


  fine-tuning the structurally pruned model
  epoch 1 | step   100 | loss 0.4024
  epoch 1 | step   200 | loss 0.2847
  epoch 1 | step   300 | loss 0.2628
  epoch 1 | step   400 | loss 0.1760
  epoch 1 finished in 321.9s

after surgery      0.5092
after fine-tuning  0.8612  (baseline 0.9140, delta -0.0528)


In [18]:
bs1 = {k: tf.constant(v[:1]) for k, v in encode(val_txt["sentence"]).items()}
dense_lat = lk.measure_latency(lk.hf_logits_fn(bert), bs1, warmup=10, runs=50, batch_size=1)
slim_lat  = lk.measure_latency(lk.hf_logits_fn(slim), bs1, warmup=10, runs=50, batch_size=1)

slim_dir = lk.model_dir("bert-sst2-ffn50", clean=True)
slim.save_pretrained(slim_dir)
tokenizer.save_pretrained(slim_dir)

print(f"dense      p50 {dense_lat['p50_ms']:6.2f} ms | {lk.size_mb(TEACHER_DIR):6.1f} MB")
print(f"structured p50 {slim_lat['p50_ms']:6.2f} ms | {lk.size_mb(slim_dir):6.1f} MB")
print(f"\nspeedup {dense_lat['p50_ms']/slim_lat['p50_ms']:.2f}x  |  "
      f"size {lk.size_mb(slim_dir)/lk.size_mb(TEACHER_DIR):.2f}x  |  "
      f"accuracy {slim_acc - bert_acc:+.4f}")

lk.record("pruned-structured-ffn50", model="BERT-base, FFN 3072->1536",
          task="binary sentiment", dataset="SST-2",
          params_m=round(slim.num_parameters()/1e6, 1),
          size_mb=round(lk.size_mb(slim_dir), 1),
          quality=round(slim_acc, 4), quality_metric="accuracy",
          p50_ms=slim_lat["p50_ms"], p95_ms=slim_lat["p95_ms"],
          throughput_rps=slim_lat["throughput_rps"], device=lk.device_label(),
          notes="structured: real shape reduction, real speedup")
lk.ledger_df()

dense      p50  10.31 ms |  439.2 MB
structured p50   7.81 ms |  325.8 MB

speedup 1.32x  |  size 0.74x  |  accuracy -0.0528


,stage,model,task,dataset,params_m,size_mb,quality,quality_metric,p50_ms,p95_ms,throughput_rps,device,notes,p50_ms_batch32,imdb_accuracy
0,baseline,bert-base-uncased (fine-tuned),binary sentiment,SST-2,109.5,439.2,0.9140,accuracy,26.96,31.34,37.1,GPU,teacher; fixed 128-token padding,291.02,NaN
1,squad-teacher,bert-base-uncased (SQuAD v1.1),extractive QA,SQuAD v1.1,108.9,435.8,89.8700,F1,30.03,31.52,33.3,GPU,12 layers; distillation teacher,NaN,NaN
2,distilled-4L,BERT 4-layer student,extractive QA,SQuAD v1.1,52.2,209.8,54.8200,F1,10.17,11.68,98.3,GPU,"T=3.0, alpha=0.7, layers copied [2, 5, 8, 11]",NaN,NaN
3,quantized-int8,"BERT-base, int8 per-channel kernels",binary sentiment,SST-2 / IMDB,109.5,181.7,0.9106,accuracy,27.82,40.08,35.9,GPU,size projected from format; accuracy measured;...,NaN,0.8265
4,quantized-fp16,"BERT-base, fp16 weights",binary sentiment,SST-2 / IMDB,109.5,219.0,0.9140,accuracy,27.82,40.08,35.9,GPU,fp16 is typically lossless for BERT-scale infe...,NaN,NaN
5,pruned-unstructured-70,"BERT-base, 70% sparse kernels",binary sentiment,SST-2,109.5,221.6,0.8119,accuracy,11.16,15.15,89.6,GPU,gzipped size; latency unchanged (dense kernels),NaN,NaN
6,pruned-structured-ffn50,"BERT-base, FFN 3072->1536",binary sentiment,SST-2,81.2,325.8,0.8612,accuracy,7.81,8.49,128.0,GPU,"structured: real shape reduction, real speedup",NaN,NaN


Compare the two pruning rows in the ledger. Unstructured pruning removed a larger fraction of the weights and delivered no latency improvement. Structured pruning removed fewer and delivered a real one, along with a genuinely smaller uncompressed artifact.

That is the whole argument. **Match the pruning granularity to what your serving stack can exploit.** If you cannot name the kernel that will take advantage of your sparsity pattern, you are optimizing storage, not speed - which is a legitimate goal, but you should know which one you are pursuing before you spend a week on it.

---
## 5. Exercises

**1. Global versus per-layer thresholds.**
Modify `compute_masks` to rank all prunable weights together against one global threshold. Compare the accuracy curve, and print the achieved sparsity per layer. Which layers does the global criterion prune hardest, and does that match your expectation from 1.4?

**2. Find the compression frontier.**
For sparsity in {0.5, 0.7, 0.8, 0.9}, run gradual pruning with fine-tuning and record gzipped size and accuracy. Plot the frontier. If your artifact registry charged by the gigabyte and your ECS cold start were dominated by model download, where would you operate?

**3. Attention head pruning.**
Extend Section 4 to prune attention heads. Score each head by the norm of its output-projection slice, drop the lowest-scoring quarter, and rebuild with a smaller `num_attention_heads`. Is the accuracy cost per parameter better or worse than FFN pruning?

**4. Stack pruning onto distillation.**
Load `student-squad-4L` from Lab 2 and prune it. Does a distilled model have less prunable redundancy than the teacher did? Whatever you find, it is a direct measurement of whether these two levers compound or compete - and Lab 5 depends on the answer.

**5. Semi-structured 2:4.**
Implement a 2:4 mask: within every group of four contiguous weights along the input dimension, keep the two largest. Measure the accuracy cost against unstructured pruning at the same 50% sparsity. You will not see a speedup on a T4 without a sparse kernel - the point is to quantify what the hardware-friendly constraint costs you in quality.

---
## 6. Wrap-up

### What you established

| Claim | Evidence |
|---|---|
| BERT tolerates ~50–70% unstructured sparsity | one-shot accuracy curve |
| Gradual pruning with fine-tuning beats one-shot | measured recovery at the same sparsity |
| Unstructured sparsity shrinks compressed size, not latency | raw vs gzipped sizes, and a ~1.0× speedup |
| Structured pruning delivers real speedup | FFN surgery, measured p50 before and after |
| TFMOT cannot wrap a subclassed transformer | the error, reproduced deliberately |

### Carry forward

1. **Decide what you are optimizing before you pick a granularity.** Storage and transfer → unstructured. Latency → structured, or 2:4 on supporting hardware.
2. **Never prune without fine-tuning** unless you are only probing the redundancy curve.
3. **Leave the embedding table alone.** Quantize it instead.
4. **Pruning is the third lever for a reason.** It has the narrowest margin and the most conditional payoff of the three.

### Checkpoint

- [ ] Ledger contains `pruned-unstructured-70` and `pruned-structured-ffn50`.
- [ ] You can state the knee of your sparsity/accuracy curve.
- [ ] You can explain to a colleague why an 85%-sparse model ran no faster.

### Next

**Lab 5 - Capstone.** We stack all three levers on one model, in the order the case study used, measuring after each step to check that the gains actually compound. The output is the artifact Day 2 packages into Docker and deploys to AWS ECS.